# Multi-Grained Spatio-Temporal Features Perceived Network for Event-Based Lip-Reading (CVPR 2022)

本 Notebook 用于在 **Google Colab** 中运行 `event-based-lip-reading` 项目，支持：
1. **环境检查与仓库克隆**：自动拉取 GitHub 项目代码并切换至工作目录。
2. **自动下载与解压数据集**：通过 `gdown` 直接从 Google Drive 下载官方 `DVS-Lip` 数据集与预训练模型权重 `MSTP`，并自动解压、整理至目标目录（同时支持挂载 Google Drive 缓存）。
3. **预训练模型测试 (Evaluation)**：一键测试官方或自定义权重的准确率（Overall Acc, Part 1 Acc, Part 2 Acc）。
4. **模型训练 (Training)**：支持自定义超参数训练与 TensorBoard 实时监控。

## 1. 检查 GPU 环境与安装依赖
请确保 Colab 运行时已设置为 GPU（`Runtime` -> `Change runtime type` -> `Hardware accelerator: GPU`）。

In [ ]:
!nvidia-smi
!pip install -q gdown tqdm tensorboard

## 2. 克隆 GitHub 仓库并进入项目目录
- 如果你是直接从 Fork 后的 GitHub 仓库在 Colab 中打开本 Notebook，请将下方的 `REPO_URL` 替换为你自己 Fork 的仓库地址（或者保持原样后再 `git remote set-url`）。
- 如果当前目录已经在仓库内，脚本会自动识别并跳过克隆。

In [ ]:
import os

# ================= 配置区 =================
# 可修改为你自己 Fork 后的 GitHub 仓库地址，例如: "https://github.com/<你的用户名>/event-based-lip-reading.git"
REPO_URL = "https://github.com/tgc1997/event-based-lip-reading.git"
PROJECT_DIR = "/content/event-based-lip-reading"
# ==========================================

if os.path.exists("main.py") and os.path.exists("model/model.py"):
    PROJECT_DIR = os.getcwd()
    print(f"当前已在项目目录中: {PROJECT_DIR}")
else:
    if not os.path.exists(PROJECT_DIR):
        print(f"正在克隆仓库: {REPO_URL} ...")
        !git clone {REPO_URL} {PROJECT_DIR}
    os.chdir(PROJECT_DIR)
    print(f"已切换工作目录至: {os.getcwd()}")

os.makedirs("data", exist_ok=True)
os.makedirs("log", exist_ok=True)
!ls -la

## 3. 自动下载并解压 DVS-Lip 数据集
官方 `DVS-Lip` 数据集存储在 Google Drive (`1dBEgtmctTTWJlWnuWxFtk8gfOdVVpkQ0`)。

这里提供两种方式：
- **方式 A（默认，推荐）**：直接使用 `gdown` 从公开 Google Drive 链接高速下载到 Colab 本地磁盘 `/content/event-based-lip-reading/data` 并自动解压。
- **方式 B（可选）**：如果你已经将数据集保存在自己的 Google Drive 中，可以开启 `USE_GOOGLE_DRIVE = True` 挂载云盘直接解压或软链接，避免每次重新下载。

In [ ]:
import os
import shutil
import zipfile
import tarfile
import gdown

# 是否挂载你自己的 Google Drive（若想将数据集缓存在自己的云盘中可设为 True）
USE_GOOGLE_DRIVE = False
GDRIVE_ARCHIVE_PATH = "/content/drive/MyDrive/DVS-Lip.zip"  # 若 USE_GOOGLE_DRIVE=True，可指定云盘中压缩包路径

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

DATASET_FILE_ID = "1dBEgtmctTTWJlWnuWxFtk8gfOdVVpkQ0"
DATA_ROOT = os.path.join(PROJECT_DIR, "data")
TARGET_DATASET_DIR = os.path.join(DATA_ROOT, "DVS-Lip")
ARCHIVE_PATH = os.path.join(DATA_ROOT, "DVS-Lip.zip")

if os.path.exists( os.path.join(TARGET_DATASET_DIR, "train") ) and os.path.exists( os.path.join(TARGET_DATASET_DIR, "test") ):
    print(f"✅ 数据集已就绪: {TARGET_DATASET_DIR}")
else:
    # 1. 下载或复制压缩包
    if USE_GOOGLE_DRIVE and os.path.exists(GDRIVE_ARCHIVE_PATH):
        print(f"发现 Google Drive 中已有数据集压缩包: {GDRIVE_ARCHIVE_PATH}")
        ARCHIVE_PATH = GDRIVE_ARCHIVE_PATH
    elif not os.path.exists(ARCHIVE_PATH):
        print("正在从 Google Drive 下载 DVS-Lip 数据集...")
        url = f"https://drive.google.com/uc?id={DATASET_FILE_ID}"
        gdown.download(url, ARCHIVE_PATH, quiet=False, fuzzy=True)
        if USE_GOOGLE_DRIVE and os.path.exists("/content/drive/MyDrive") and not os.path.exists(GDRIVE_ARCHIVE_PATH):
            print(f"正在备份数据集压缩包到 Google Drive: {GDRIVE_ARCHIVE_PATH}")
            shutil.copy2(ARCHIVE_PATH, GDRIVE_ARCHIVE_PATH)

    # 2. 自动识别压缩格式并解压到 ./data/
    print(f"正在解压 {ARCHIVE_PATH} 到 {DATA_ROOT} ...")
    if zipfile.is_zipfile(ARCHIVE_PATH):
        with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zf:
            zf.extractall(DATA_ROOT)
    elif tarfile.is_tarfile(ARCHIVE_PATH):
        with tarfile.open(ARCHIVE_PATH, 'r:*') as tf:
            tf.extractall(DATA_ROOT)
    else:
        # 使用系统命令兜底解压 (支持 7z / rar / zip / tar)
        !unzip -q -o "{ARCHIVE_PATH}" -d "{DATA_ROOT}" || tar -xf "{ARCHIVE_PATH}" -C "{DATA_ROOT}"

    # 3. 检查解压后的目录结构并修正（如果解压后 train/test 直接在 data/ 下或嵌套在其他目录中）
    if not os.path.exists(TARGET_DATASET_DIR):
        if os.path.exists(os.path.join(DATA_ROOT, "train")) and os.path.exists(os.path.join(DATA_ROOT, "test")):
            os.makedirs(TARGET_DATASET_DIR, exist_ok=True)
            shutil.move(os.path.join(DATA_ROOT, "train"), os.path.join(TARGET_DATASET_DIR, "train"))
            shutil.move(os.path.join(DATA_ROOT, "test"), os.path.join(TARGET_DATASET_DIR, "test"))

    # 4. 删除本地临时压缩包以节省 Colab 磁盘空间
    if ARCHIVE_PATH == os.path.join(DATA_ROOT, "DVS-Lip.zip") and os.path.exists(ARCHIVE_PATH):
        os.remove(ARCHIVE_PATH)
        print("已清理临时压缩包 DVS-Lip.zip，释放磁盘空间。")

# 验证数据集结构
train_classes = len(os.listdir(os.path.join(TARGET_DATASET_DIR, "train"))) if os.path.exists(os.path.join(TARGET_DATASET_DIR, "train")) else 0
test_classes = len(os.listdir(os.path.join(TARGET_DATASET_DIR, "test"))) if os.path.exists(os.path.join(TARGET_DATASET_DIR, "test")) else 0
print(f"数据集校验完成 -> 训练集类别数: {train_classes}, 测试集类别数: {test_classes}")

## 4. 下载官方预训练模型权重 (MSTP)
官方提供了预训练好的 `MSTP` 模型权重（Google Drive 文件夹 ID: `1xi9qoQ0LjEoo6SvWOH2pSXrdjia9_jJC`），下载后存放在 `log/mstp/model_best.pth` 即可直接运行评估测试。

In [ ]:
import os
import glob
import shutil
import gdown

PRETRAINED_FOLDER_URL = "https://drive.google.com/drive/folders/1xi9qoQ0LjEoo6SvWOH2pSXrdjia9_jJC"
LOG_ROOT = os.path.join(PROJECT_DIR, "log")
MSTP_LOG_DIR = os.path.join(LOG_ROOT, "mstp")
os.makedirs(MSTP_LOG_DIR, exist_ok=True)

if os.path.exists(os.path.join(MSTP_LOG_DIR, "model_best.pth")):
    print(f"✅ 预训练权重已存在: {os.path.join(MSTP_LOG_DIR, 'model_best.pth')}")
else:
    print("正在从 Google Drive 下载预训练模型权重...")
    try:
        gdown.download_folder(url=PRETRAINED_FOLDER_URL, output=LOG_ROOT, quiet=False, use_cookies=False)
        # 如果下载下来的文件直接落在 log/ 下，或者文件夹大小写不一致，自动整理到 log/mstp/model_best.pth
        pth_files = glob.glob(os.path.join(LOG_ROOT, "**", "*.pth"), recursive=True)
        if pth_files and not os.path.exists(os.path.join(MSTP_LOG_DIR, "model_best.pth")):
            shutil.copy2(pth_files[0], os.path.join(MSTP_LOG_DIR, "model_best.pth"))
        print("✅ 预训练模型下载完成！")
    except Exception as e:
        print(f"⚠️ 自动下载预训练文件夹失败 ({e})，若仅需从头训练可忽略此步骤。")

!ls -lh log/ log/mstp/ 2>/dev/null || true

## 5. 测试模型 (Evaluation)
- **注意**：根據 README 与 `run.sh`，请确保测试时的 `--num_bins`、`--alpha`、`--beta` 与权重训练时的配置一致：
  - 配置 1 (README 测试命令): `--num_bins=1+7 --alpha=4 --beta=7`
  - 配置 2 (`run.sh` 配置): `--num_bins=1+4 --alpha=4 --beta=4`
- 在 Colab 免费版 (2 核 CPU) 环境中，建议将 `--num_workers` 设为 `2` 或 `4`，避免 DataLoader 多进程卡顿。

In [ ]:
# 测试预训练模型 (若权重对应 num_bins=1+4, alpha=4, beta=4，可相应修改参数)
!python main.py \
    --gpus=0 \
    --num_bins=1+7 \
    --test=True \
    --alpha=4 \
    --beta=7 \
    --batch_size=32 \
    --num_workers=2 \
    --weights=mstp

## 6. 训练模型 (Training)
运行下方单元格开始从头训练 `MSTP` 模型。
- 训练日志保存在 `log/<log_dir>/train_log.txt`
- 最优模型权重自动保存在 `log/<log_dir>/model_best.pth`
- TensorBoard 记录保存在 `log/<log_dir>/record/`

In [ ]:
# 启动 TensorBoard 面板（可在训练过程中点击刷新查看 Loss 和 Acc 曲线）
%load_ext tensorboard
%tensorboard --logdir log

In [ ]:
# 开始训练模型
!python main.py \
    --gpus=0 \
    --num_bins=1+4 \
    --test=False \
    --alpha=4 \
    --beta=4 \
    --lr=3e-4 \
    --batch_size=32 \
    --max_epoch=80 \
    --num_workers=2 \
    --log_dir=exp_colab

## 7. 测试自己训练好的模型 & 备份权重到 Google Drive
训练完成后，可以直接评估刚刚保存在 `log/exp_colab/model_best.pth` 的最优模型，并将其复制到 Google Drive 防止 Colab 断开后丢失。

In [ ]:
# 测试刚训练完成的模型 (exp_colab)
!python main.py \
    --gpus=0 \
    --num_bins=1+4 \
    --test=True \
    --alpha=4 \
    --beta=4 \
    --batch_size=32 \
    --num_workers=2 \
    --weights=exp_colab

In [ ]:
# （可选）将训练日志和权重备份到自己的 Google Drive
BACKUP_TO_DRIVE = False

if BACKUP_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    dst_dir = "/content/drive/MyDrive/event-based-lip-reading-logs"
    shutil.copytree(os.path.join(PROJECT_DIR, "log"), dst_dir, dirs_exist_ok=True)
    print(f"✅ 已将所有训练日志与权重备份至: {dst_dir}")